In [14]:
from Bio.PDB import PDBParser
from rdkit import Chem

import numpy as np

import torch
import torch_cluster
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GINEConv, knn_graph

# Load protein


In [15]:
parser = PDBParser()

pdb_code = '1a30'

structure = parser.get_structure("protein", f"./data/PDBBind/{pdb_code}/{pdb_code}_protein.pdb")
structure

<Structure id=protein>

In [16]:
AMINO_ACIDS = [
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
]
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

coords, features = [], []

for residue in structure.get_residues():
    # skip non aminoacids
    if residue.get_id()[0] != ' ':
        continue

    res_name = residue.get_resname()

    # skip unknown aminoacids
    if res_name not in AA_TO_IDX:
        continue

    ca = residue['CA'].get_vector().get_array()
    one_hot = np.zeros(len(AMINO_ACIDS))
    one_hot[AA_TO_IDX[res_name]] = 1.0

    coords.append(ca)
    features.append(one_hot)

coords = torch.tensor(np.array(coords), dtype=torch.float32)    # (N, 3)
features = torch.tensor(np.array(features), dtype=torch.float32)    # (N, 20)

coords.shape, features.shape

(torch.Size([198, 3]), torch.Size([198, 20]))

# Define model

In [17]:
class PocketFinder(nn.Module):
    def __init__(self, in_dim: int = 20, hidden: int = 64):
        super().__init__()

        # from one-hot encoding to embedding
        self.embedding = nn.Linear(in_dim, hidden)

        self.conv = GINEConv(nn = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        ), edge_dim=1)

        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, h, pos, batch):
        h = self.embedding(h)

        edge_index = knn_graph(pos, k=10, batch=batch, loop=False)
        row, col = edge_index
        edge_attr = (pos[row] - pos[col]).norm(dim=-1, keepdim=True)

        h = self.conv(h, edge_index, edge_attr)

        h = self.head(h)

        return h.squeeze(-1)

In [18]:
model = PocketFinder(in_dim = len(AMINO_ACIDS))
model

PocketFinder(
  (embedding): Linear(in_features=20, out_features=64, bias=True)
  (conv): GINEConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  ))
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

#### Test model (random data)

In [19]:
pos = torch.randn(50, 3)
h = torch.zeros(50, 20)
batch = torch.zeros(50, dtype=torch.long)

out = model(h, pos, batch)
print(out.shape)

torch.Size([50])


In [20]:
probs = model(h, pos, batch).sigmoid()
print(probs.shape)
print(probs.min().item(), probs.max().item())

torch.Size([50])
0.5686992406845093 0.5832030773162842


# Loading ligand


In [21]:
mol = Chem.MolFromMol2File(f'./data/PDBBind/{pdb_code}/{pdb_code}_ligand.mol2')
conf = mol.GetConformer()

ligand_coords = torch.tensor(conf.GetPositions(), dtype=torch.float32)  # (L, 3)

print(ligand_coords.shape)

torch.Size([26, 3])


# Generate pocket labels


In [22]:
POCKET_DIST_CUTOFF = 6.5

pairwise_dist_mat = torch.cdist(coords, ligand_coords)  # (N, L)
print(pairwise_dist_mat.shape)

N, L = pairwise_dist_mat.shape

pocket_labels = (pairwise_dist_mat.min(dim=1).values < POCKET_DIST_CUTOFF).float()  # (N,)
print(pocket_labels.sum().item(), pocket_labels.shape)


torch.Size([198, 26])
21.0 torch.Size([198])


#### Test model (single protein)

In [23]:
batch = torch.zeros(N)

logits = model(features, coords, batch)

print(logits.shape) # (N,)

torch.Size([198])


# Define Loss


In [24]:
loss = F.binary_cross_entropy_with_logits(logits, pocket_labels,
                                          pos_weight=torch.tensor(9))

print(loss.item())

1.2955880165100098


# Training


In [207]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

optimizer.zero_grad()
logits = model(features, coords, batch)
loss = F.binary_cross_entropy_with_logits(logits, pocket_labels,
                                          pos_weight=torch.tensor(9))
loss.backward()
optimizer.step()

print(loss.item())

1.2569226026535034


In [208]:
with torch.no_grad():
    probs = model(features, coords, batch).sigmoid()

print("predicted pocket residues:", (probs > 0.5).sum().item())
print("actual pocket residues:", pocket_labels.sum().item())
print("max prob:", probs.max().item())
print("min prob:", probs.min().item())

predicted pocket residues: 102
actual pocket residues: 21.0
max prob: 0.5312031507492065
min prob: 0.45573973655700684
